# CAMELYON, opened cold

This notebook opens the CAMELYON store in Arraylake with nothing but the store itself: no bucket listing, no challenge CSVs, no ingest code. Everything below comes from the store's own metadata.

The repo is `earthmover-biology/camelyon`. You need an Arraylake login with read access (`al auth login`), and `imagecodecs` and `imagecodecs-zarr` to decode the JPEG and LZW tiles. The pixels stay in the challenges' public bucket, `s3://camelyon-dataset`; the store references them, and Arraylake authorizes those reads.

In [ ]:
import json

import imagecodecs.zarr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
import zarr
from arraylake import Client

imagecodecs.zarr.register_codecs()
repo = Client().get_repo("earthmover-biology/camelyon")
root = zarr.open_group(repo.readonly_session("main").store, mode="r")
about = dict(root.attrs)
print(about["title"], "\n")
print(about["description"], "\n")
print("license:", about["license"])
print("needs:", about["requires"])

The history shows the store growing: the CAMELYON16 slides first, then CAMELYON17 in batches of 100 slides.

In [ ]:
for snapshot in list(repo.ancestry(branch="main"))[:15]:
    print(snapshot.written_at.date(), snapshot.id[:12], snapshot.message[:110])

## The two challenges

Each challenge is an OME-Zarr collection. Its `annotations` attribute is the slide table, one column per field.

In [ ]:
for name in ("CAMELYON16", "CAMELYON17"):
    print(f"{name}: {root[name].attrs['description']}\n")

c16 = pd.DataFrame(root["CAMELYON16"].attrs["annotations"]).set_index("slide")
c17 = pd.DataFrame(root["CAMELYON17"].attrs["annotations"]).set_index("slide")
c16.groupby(["split", "type", "class"]).size()

CAMELYON17 labels each patient with a pN stage and each slide with a metastasis category, for the 100 training patients. Its `center_names` attribute maps the center numbers to hospitals, with the evidence for the mapping beside it.

In [ ]:
c17_attrs = dict(root["CAMELYON17"].attrs)
patients = pd.DataFrame(c17_attrs["patients"]).set_index("group")
print(c17_attrs["center_names"], "\n")
print(c17_attrs["center_names_decision"], "\n")
pd.crosstab(c17.center, c17.stage)

## One slide, its mask and its outlines

Pick the CAMELYON17 slide with outlines from the center whose scanner wrote Philips TIFFs. Each slide is a group under its patient, with `image` (the scan), `image/labels/mask` (the reference mask) and `shapes` (the pathologists' outlines).

In [ ]:
slide = c17[c17.has_shapes & (c17.center == "4")].index[0]
row = c17.loc[slide]
group = root[f"CAMELYON17/{row.patient}/node_{row.node}"]
image = group["image"]
levels = [d["path"] for d in image.attrs["ome"]["multiscales"][0]["datasets"]]
print(group.attrs["description"])
print("pixel size:", image.attrs["pixel_size"]["um"], "micrometer")
for path in levels:
    print(f"level {path}: shape {image[path].shape}, chunks {image[path].chunks}")

The outlines are stored twice: as arrays that rebuild the ASAP XML byte for byte, and as GeoJSON in QuPath's feature schema. The GeoJSON is in level-0 pixels, shifted by half a pixel into QuPath's frame, so dividing by a level's downsampling factor puts it on that level.

In [ ]:
level = levels[-3]
scan = np.moveaxis(image[level][:], 0, -1)
mask = group[f"image/labels/mask/{level}"][:]
factor = image["0"].shape[2] / image[level].shape[2]
features = json.loads(bytes(group["shapes/geojson"][:]))["features"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(scan)
axes[1].imshow(mask, cmap="viridis", vmin=0, vmax=2, interpolation="nearest")
for ax in axes:
    for feature in features:
        geometry = shapely.geometry.shape(feature["geometry"])
        for polygon in getattr(geometry, "geoms", [geometry]):
            ax.plot(*(np.asarray(polygon.exterior.coords) / factor).T, color="red", lw=1)
    ax.set_axis_off()
axes[0].set_title(f"{slide}, level {level}")
axes[1].set_title("reference mask: 0 background, 1 normal tissue, 2 tumor");

## Full resolution under one outline

A region of level 0 reads only the 512 x 512 JPEG tiles it touches, from the slide's TIFF in the bucket.

In [ ]:
largest = max((shapely.geometry.shape(f["geometry"]) for f in features), key=lambda g: g.area)
x, y = (int(v) for v in largest.centroid.coords[0])
window = np.moveaxis(image["0"][:, y - 512 : y + 512, x - 512 : x + 512], 0, -1)
plt.figure(figsize=(7, 7))
plt.imshow(window)
plt.title(f"level 0, 1024 x 1024 pixels around ({x}, {y})")
plt.axis("off");

## What the files said

Every image keeps its TIFF pages' tags, and each challenge keeps its own text files verbatim.

In [ ]:
tags = image.attrs["tiff_tags"][0]
print({k: v for k, v in tags.items() if k != "ImageDescription"})
print("source:", image.attrs["source_url"])
print("files kept verbatim:", list(c17_attrs["source_files"]))
print(c17_attrs["source_files"]["README.md"][:600])